In [1]:
import urllib.request, subprocess, platform, json
from pathlib import Path

DATA = Path("data/hard")
DATA.mkdir(parents=True, exist_ok=True)
rar = DATA / "unbalanced-reviews.rar"

if not rar.exists():
    api = "https://api.github.com/repos/elnagara/HARD-Arabic-Dataset/contents/data"
    files = json.load(urllib.request.urlopen(api))
    target = next(f for f in files if f["name"].lower().startswith("unbalanced") and f["name"].endswith(".rar"))
    urllib.request.urlretrieve(target["download_url"], rar)

print(rar.name, "|", rar.stat().st_size // 1_000_000, "MB")

unbalanced-reviews.rar | 23 MB


In [2]:
txt = DATA / "unbalanced-reviews.txt"
if not txt.exists():
    if platform.system() == "Windows":
        cmd = ["tar", "-xf", str(rar), "-C", str(DATA)]
    else:
        cmd = ["unrar", "x", "-o+", str(rar), str(DATA) + "/"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print("ok" if r.returncode == 0 else r.stderr)
print(txt.name, txt.stat().st_size // 1_000_000, "MB")

unbalanced-reviews.txt 159 MB


In [3]:
import pandas as pd

path = DATA / "unbalanced-reviews.txt"
head_bytes = path.read_bytes()[:4]
enc = "utf-16" if head_bytes[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig"
df = pd.read_csv(path, sep="\t", encoding=enc, quoting=3)
print("file:", path.name, "| encoding:", enc, "| shape:", df.shape)

file: unbalanced-reviews.txt | encoding: utf-16 | shape: (409562, 7)


In [4]:
AUDIT = DATA / "audit"
AUDIT.mkdir(exist_ok=True)
lab_path = AUDIT / "to_label.csv"

if lab_path.exists():
    print("to_label.csv exists -> not overwritten")
else:
    sample = df.groupby("rating", group_keys=False).sample(n=40, random_state=42)
    sample = sample.sample(frac=1, random_state=42)
    sample[["no", "review"]].assign(my_label="", note="").to_csv(lab_path, index=False, encoding="utf-8-sig")
    sample[["no", "rating"]].to_csv(AUDIT / "answer_key.csv", index=False)
    print(len(sample), "reviews |", sample["rating"].value_counts().sort_index().to_dict())

to_label.csv exists -> not overwritten


In [5]:
lab_path = AUDIT / "to_label.csv"
lab = pd.read_csv(lab_path, encoding="utf-8-sig", dtype=str).fillna("")
todo = lab.index[lab["my_label"] == ""]
print(f"left: {len(todo)} / {len(lab)}  |  p / n / m  (+ space + note, optional)  |  q = stop")

for i in todo:
    print("\n" + "=" * 70)
    print(f"[{i + 1}/{len(lab)}]  no = {lab.at[i, 'no']}\n")
    print(lab.at[i, "review"])
    ans = input("label: ").strip()
    if ans.lower() == "q":
        break
    label, _, note = ans.partition(" ")
    while label not in ("p", "n", "m"):
        label, _, note = input("only p / n / m: ").strip().partition(" ")
    lab.at[i, "my_label"], lab.at[i, "note"] = label, note
    lab.to_csv(lab_path, index=False, encoding="utf-8-sig")

print("\ndone:", (lab["my_label"] != "").sum(), "/", len(lab))

left: 0 / 200  |  p / n / m  (+ space + note, optional)  |  q = stop

done: 200 / 200


In [6]:
lab = pd.read_csv(AUDIT / "to_label.csv", encoding="utf-8-sig").fillna("")
key = pd.read_csv(AUDIT / "answer_key.csv")
m = lab.merge(key, on="no")
m["rating_label"] = m["rating"].map(lambda r: "n" if r <= 3 else "p")

print(pd.crosstab(m["rating"], m["my_label"], margins=True))

clear = m[m["my_label"] != "m"]
mismatch = clear["my_label"] != clear["rating_label"]
print(f"\nclear reviews: {len(clear)} | mismatch: {mismatch.sum()} ({mismatch.mean():.0%})")
print("\nmismatch per rating:\n", clear[mismatch].groupby("rating").size())

print("\nexamples:")
for _, r in clear[mismatch].head(8).iterrows():
    print(f"\n[rating {r['rating']} | you: {r['my_label']}] {r['note']}\n{r['review'][:300]}")

my_label   m   n   p  All
rating                   
1          1  37   2   40
2          2  38   0   40
3         22  14   4   40
4         16   9  15   40
5          8   0  32   40
All       49  98  53  200

clear reviews: 151 | mismatch: 15 (10%)

mismatch per rating:
 rating
1    2
3    4
4    9
dtype: int64

examples:

[rating 4 | you: n] 
رائع. القهوة عند المدخلالمدخل الفاره. الأثاث قديم و دورة المياه تحتاج الى مزيد من التحسينات

[rating 3 | you: p] 
مرضي. الفندق يعتبر جيد جداً مقارنه بسعرة وأشكر موظف الاستقبال/ سلمان و الأخ / محمد كانا جداً متعاونين وخدومين. موقع الفندق + بعد وقوف الباص من الحرم نمشي قرابة 1 كيلو متر

[rating 4 | you: n] 
جيد. الموقع والنظافه. تم سحب المبلغ مرتين زحتى هذه الحظة لم تسترجع

[rating 3 | you: p] 
مرضي. . مساحات المواقف

[rating 3 | you: p] 
“الاقامة كانت ممتعة”. الموقع وخدمات المطعم. الانترنت

[rating 1 | you: p] 
“اقامه في فندق ب ٢٠٠ ريال يضاهي هذا الفندق بكل شي”. لاشي. المصداقيه

[rating 4 | you: n] 
“للافضل”. قام أحد في الشقق المجاورة بازعاجنا طوا

In [7]:
counts = df["rating"].value_counts().sort_index()
print(pd.DataFrame({"count": counts, "share": (counts / len(df)).round(3)}))
print("\ntotal:", len(df))

         count  share
rating               
1        14382  0.035
2        38467  0.094
3        80326  0.196
4       132208  0.323
5       144179  0.352

total: 409562


In [8]:
first = df["review"].astype(str).str.split(".", n=1).str[0].str.strip()

top = first.value_counts().head(20)
print(top)

print("\n", pd.crosstab(first[first.isin(top.index[:12])], df["rating"]))

review
استثنائي       37543
جيد            34467
مرضي           16954
ممتاز          13901
مقبول          12991
جيد جداً       11801
رائع           11716
مخيب للأمل      8790
حسن             6798
“ممتاز”         5964
ضعيف            5884
ضعيف جداً       4869
“جيد”           4482
“رائع”          3185
"جيد            2767
"استثنائي       2488
“جيد جدا”       1996
“فندق رائع”     1445
"مرضي           1417
“مقبول”         1372
Name: count, dtype: int64

 rating         1     2      3      4      5
review                                     
استثنائي       0     0      0      0  37543
جيد            0     0      0  34467      0
جيد جداً       0     0      0  11801      0
حسن            0     0   6798      0      0
رائع           0     0      0  11716      0
ضعيف           0  5884      0      0      0
ضعيف جداً   4869     0      0      0      0
مخيب للأمل     0  8790      0      0      0
مرضي           0     0  16954      0      0
مقبول          0     0  12991      0      0
ممتاز          0 

In [9]:
first = df["review"].astype(str).str.split(".", n=1).str[0].str.strip()
stats = df.assign(first=first).groupby("first")["rating"].agg(["count", "nunique"])
site_labels = stats[(stats["nunique"] == 1) & (stats["count"] >= 500)].index.tolist()
print("site labels:", site_labels)

is_site = first.isin(site_labels)
rest = df["review"].astype(str).str.split(".", n=1).str[1].fillna("").str.strip()
df["review_clean"] = df["review"].astype(str).where(~is_site, rest)

print(f"\nremoved from {is_site.sum()} reviews ({is_site.mean():.0%})")
still = df["review_clean"].str.split(".", n=1).str[0].str.strip().isin(site_labels)
print("still starting with a site label:", still.sum())
print("\n", df.loc[is_site, ["rating", "review", "review_clean"]].head(3).to_string())

site labels: ['"استثنائي', '"جيد', '"جيد جداً', '"حسن', '"رائع', '"مخيب للأمل', '"مرضي', '"مقبول', '"ممتاز', 'استثنائي', 'جيد', 'جيد جداً', 'حسن', 'رائع', 'ضعيف', 'ضعيف جداً', 'مخيب للأمل', 'مرضي', 'مقبول', 'ممتاز']

removed from 177737 reviews (43%)
still starting with a site label: 946

    rating                                                 review                                     review_clean
2       5     استثنائي. سهولة إنهاء المعاملة في الاستقبال. لاشيئ         سهولة إنهاء المعاملة في الاستقبال. لاشيئ
3       4                                   جيد جداً. 😊👍. لايوجد                                       😊👍. لايوجد
6       3  مرضي. . حمام السباحه لايعمل و الاضاءه في الغرفه خافته  . حمام السباحه لايعمل و الاضاءه في الغرفه خافته


In [10]:
print(df.loc[still, ["rating", "review"]].head(8).to_string())

      rating                                                  review
406        4                                          جيد. جيد. لاشئ
868        5                                   استثنائي. ممتاز. لاشي
1275       4                                  جيد جداً. ممتاز. لا شى
2274       4                                       جيد. ممتاز. لا شي
2630       5                                        استثنائي. رائع. 
2651       3  مرضي. ممتاز. الفندق ممتاز بصفه عام بس غالي الثمن قليلا
3250       3                    مرضي. جيد. لا يوجد جهاز لتسخين الاكل
4272       5                                 استثنائي. ممتاز. لايوجد


In [11]:
n_dots = df["review_clean"].str.count(r"\.").clip(upper=6)
print(pd.crosstab(n_dots, is_site.rename("site_label_removed"), normalize="columns").round(3))

site_label_removed  False  True 
review_clean                    
1                   0.000  0.856
2                   0.794  0.046
3                   0.055  0.038
4                   0.048  0.017
5                   0.026  0.014
6                   0.077  0.029


In [12]:
latin = df["review_clean"].str.count(r"[A-Za-z]")
letters = df["review_clean"].str.count(r"[A-Za-z\u0600-\u06FF]").clip(lower=1)
latin_share = latin / letters

print("reviews with any English letter:", (latin > 0).sum())
print("reviews mostly English (>50%):  ", (latin_share > 0.5).sum())
print("\nexamples, mostly English:")
print(df.loc[latin_share > 0.5, "review_clean"].head(5).to_string())

reviews with any English letter: 9937
reviews mostly English (>50%):   1341

examples, mostly English:
862                “mohamed”. فندق رائع جدا جدا. Nothaing
1364    "“Excellent quite hotel.”. النظافة عالية والمو...
1420    Very quiet place. Appropriate for relaxation a...
1834    “فندق الصراصير و الإزعاج....cockroach and nois...
2073    النظافه. transportation not  what the written ...


In [13]:
meta = ["Hotel name", "rating", "user type", "room type", "nights", "review"]
print("same review recorded twice (all columns):", df.duplicated(subset=meta).sum())
print("same text only (any hotel/user):          ", df.duplicated(subset=["review_clean"]).sum())

print("\nmost repeated texts:")
print(df["review_clean"].value_counts().head(8))

same review recorded twice (all columns): 188
same text only (any hotel/user):           13110

most repeated texts:
review_clean
كل شي. لا شي      515
كل شي.            390
الموقع.           303
كل شي. لاشي       303
ممتاز.            221
كل شيء. لا شيء    198
نعم.              158
كل شيء.           150
Name: count, dtype: int64


In [14]:
before = len(df)

clean = df.drop_duplicates(subset=meta, keep="first").copy()
n_dup = before - len(clean)

core = clean["review_clean"].str.replace(r"[\s\.\،,!؟?:;\-\"“”'()]", "", regex=True)
empty = core.str.len() == 0
n_empty = empty.sum()
clean = clean[~empty]

clean["label"] = clean["rating"].map({1: "neg", 2: "neg", 3: "neu", 4: "pos", 5: "pos"})

print(f"start: {before} | duplicates: -{n_dup} | empty: -{n_empty} | final: {len(clean)}")
print("\n", clean["label"].value_counts(normalize=True).round(3))
print("\nexamples of removed empty reviews:")
print(df.loc[df.index.isin(core[core.str.len() == 0].index), "review"].head(5).to_string())

start: 409562 | duplicates: -188 | empty: -62 | final: 409312

 label
pos    0.675
neu    0.196
neg    0.129
Name: proportion, dtype: float64

examples of removed empty reviews:
3556                      استثنائي. . -----
18425                             حسن. .. .
24277                             “-”. -. -
28688                      “...”. .... ....
32351    “....”. ............ .............


In [15]:
import numpy as np

rng = np.random.default_rng(42)
hotels = np.array(clean["Hotel name"].unique(), dtype=object)
rng.shuffle(hotels)
print("unique hotels:", len(hotels), "| after shuffle still unique:", len(set(hotels)) == len(hotels))

sizes = clean["Hotel name"].value_counts().reindex(hotels)
cum = sizes.cumsum() / sizes.sum()
split_of = pd.Series(np.where(cum <= 0.8, "train", np.where(cum <= 0.9, "val", "test")), index=hotels)
clean["split"] = clean["Hotel name"].map(split_of)

print("reviews per split:\n", clean["split"].value_counts(normalize=True).round(3))
print("\nhotels per split:\n", clean.groupby("split")["Hotel name"].nunique())
print("\nlabel share per split:\n", pd.crosstab(clean["split"], clean["label"], normalize="index").round(3))

h = clean.groupby("split")["Hotel name"].apply(set)
print("\nhotel overlap train-val:", len(h["train"] & h["val"]),
      "| train-test:", len(h["train"] & h["test"]),
      "| val-test:", len(h["val"] & h["test"]))
print("reviews without a split:", clean["split"].isna().sum())

unique hotels: 1173 | after shuffle still unique: True
reviews per split:
 split
train    0.800
test     0.103
val      0.098
Name: proportion, dtype: float64

hotels per split:
 split
test     115
train    949
val      109
Name: Hotel name, dtype: int64

label share per split:
 label    neg    neu    pos
split                     
test   0.143  0.208  0.649
train  0.129  0.196  0.675
val    0.118  0.182  0.700

hotel overlap train-val: 0 | train-test: 0 | val-test: 0
reviews without a split: 0


In [16]:
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sentencepiece as spm

train = clean[clean["split"] == "train"]

MODELS = Path("models")
MODELS.mkdir(exist_ok=True)
spm_prefix = MODELS / "spm_reviews_16k"

if not (MODELS / "spm_reviews_16k.model").exists():
    corpus = DATA / "train_corpus.txt"
    corpus.write_text("\n".join(train["review_clean"]), encoding="utf-8")
    spm.SentencePieceTrainer.train(
        input=str(corpus), model_prefix=str(spm_prefix),
        vocab_size=16000, model_type="bpe", character_coverage=0.9999,
    )

sp = spm.SentencePieceProcessor(model_file=str(spm_prefix) + ".model")

for text in ["بالفندق", "الفندق", "وبفنادقهم", "الموظفين محترمين", "مو نظيف أبداً", "الغرفة 😊👍"]:
    print(f"{text:<20} -> {sp.encode(text, out_type=str)}")

NameError: name 'clean' is not defined

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import time

clean["tokens"] = [" ".join(sp.encode(t, out_type=str)) for t in clean["review_clean"]]
train = clean[clean["split"] == "train"]
val = clean[clean["split"] == "val"]

vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                      ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_tr = vec.fit_transform(train["tokens"])
X_va = vec.transform(val["tokens"])
print("vocab size:", len(vec.vocabulary_), "| train:", X_tr.shape, "| val:", X_va.shape)

t0 = time.time()
clf = LogisticRegression(class_weight="balanced", max_iter=1000)
clf.fit(X_tr, train["label"])
print(f"trained in {time.time() - t0:.0f}s")

pred = clf.predict(X_va)
labels = ["neg", "neu", "pos"]
print("\n", classification_report(val["label"], pred, labels=labels, digits=3))
print("confusion matrix (rows = true, cols = predicted):")
print(pd.DataFrame(confusion_matrix(val["label"], pred, labels=labels), index=labels, columns=labels))

names = vec.get_feature_names_out()
for i, c in enumerate(clf.classes_):
    top = clf.coef_[i].argsort()[-15:][::-1]
    print(f"\ntop pieces for {c}:", " | ".join(names[top]))

vocab size: 255989 | train: (327264, 255989) | val: (40024, 255989)
trained in 48s

               precision    recall  f1-score   support

         neg      0.571     0.653     0.609      4720
         neu      0.403     0.576     0.474      7281
         pos      0.922     0.797     0.855     28023

    accuracy                          0.740     40024
   macro avg      0.632     0.675     0.646     40024
weighted avg      0.786     0.740     0.757     40024

confusion matrix (rows = true, cols = predicted):
      neg   neu    pos
neg  3082  1341    297
neu  1501  4192   1588
pos   814  4865  22344

top pieces for neg: ▁" ضعيف | ضعيف | . ▁كل | ضعيف . | ▁سيئة | ▁سيء | ▁غير | ▁سيئه | ▁. ▁كل | ▁سئ | ▁عدم | ▁لا | ▁سيئ | سيئ | ▁لايستحق

top pieces for neu: ▁ما ”. | . ▁كل | شكرا ”. | ▁فقط ”. | مقبول | ▁القيمة ”. | ▁والنظافة . | ▁عادي | ▁عدم ▁سرعة | فندق ▁جيد | ▁جيد | . ▁النظافة | فندق ▁عادي | " | ▁“ الانترنت

top pieces for pos: ▁ممتاز | ▁رائع | . ▁لا | ▁ممتازة | ▁ممتازه | ▁راقي | ▁رائعة |

In [ ]:
SITE_LABELS = ["استثنائي", "ممتاز", "رائع", "جيد جداً", "جيد", "حسن",
               "مرضي", "مقبول", "مخيب للأمل", "ضعيف", "ضعيف جداً"]

raw = df["review"].astype(str)
unq = raw.str.lstrip('"').str.strip()
first = unq.str.split(".", n=1).str[0].str.strip()
rest = unq.str.split(".", n=1).str[1].fillna("").str.strip().str.rstrip('"').str.strip()

is_site = first.isin(SITE_LABELS)
df["review_clean"] = raw.where(~is_site, rest)

print(f"removed from {is_site.sum()} reviews ({is_site.mean():.0%})")
print("\nsite label vs rating:")
print(pd.crosstab(first[is_site], df.loc[is_site, "rating"]))

print("\nremoved share per rating:")
print(is_site.groupby(df["rating"]).mean().round(3))

removed from 178512 reviews (44%)

site label vs rating:
rating         1     2      3      4      5
review                                     
استثنائي       0     0      0      0  40031
جيد            0     0      0  37234      0
جيد جداً       0     0      0  12751      0
حسن            0     0   7354      0      0
رائع           0     0      0  12682      0
ضعيف           0  6360      0      0      0
ضعيف جداً   5168     0      0      0      0
مخيب للأمل     0  9521      0      0      0
مرضي           0     0  18371      0      0
مقبول          0     0  14005      0      0
ممتاز          0     0      0      0  15035

removed share per rating:
rating
1    0.359
2    0.413
3    0.495
4    0.474
5    0.382
Name: review, dtype: float64


## Pipeline
Exploration above is kept as a record. Everything below runs top-to-bottom from reviews_pipeline.py.

In [2]:
import importlib, json, time
from pathlib import Path
import numpy as np
import pandas as pd

import reviews_pipeline as rp
importlib.reload(rp)

DATA = Path("data/hard")
MODELS = Path("models")
MODELS.mkdir(exist_ok=True)

In [3]:
import importlib, json
import reviews_pipeline as rp
importlib.reload(rp)

raw = rp.load_raw(DATA / "unbalanced-reviews.txt")
site_labels = rp.find_site_labels(raw)
(DATA / "site_labels.json").write_text(json.dumps(site_labels, ensure_ascii=False, indent=1), encoding="utf-8")
print("site labels:", len(site_labels), site_labels)

data = rp.strip_site_labels(raw, site_labels)
print("\nremoved share per rating:\n", data.groupby("rating")["had_site_label"].mean().round(3))

n0 = len(data)
data = rp.clean(data)
print(f"\nclean: {n0} -> {len(data)}")

data = rp.split_by_hotel(data)
h = data.groupby("split")["Hotel name"].apply(set)
print("\nlabel share per split:\n", pd.crosstab(data["split"], data["label"], normalize="index").round(3))
print("hotel overlap:", len(h["train"] & h["val"]), len(h["train"] & h["test"]), len(h["val"] & h["test"]))

site labels: 11 ['استثنائي', 'جيد', 'جيد جداً', 'حسن', 'رائع', 'ضعيف', 'ضعيف جداً', 'مخيب للأمل', 'مرضي', 'مقبول', 'ممتاز']

removed share per rating:
 rating
1    0.359
2    0.413
3    0.495
4    0.474
5    0.382
Name: had_site_label, dtype: float64

clean: 409562 -> 409312

label share per split:
 label    neg    neu    pos
split                     
test   0.143  0.208  0.649
train  0.129  0.196  0.675
val    0.118  0.182  0.700
hotel overlap: 0 0 0


In [4]:
MODELS = Path("models")
MODELS.mkdir(exist_ok=True)

sp2 = rp.train_tokenizer(data.loc[data["split"] == "train", "review_clean"], MODELS / "spm_reviews_16k_v2")
data = rp.add_tokens(data, sp2)
print(data[["review_clean", "tokens"]].head(3).to_string())

KeyboardInterrupt: 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import time

tr = data[data["split"] == "train"]
va = data[data["split"] == "val"]

vec2 = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                       ngram_range=(1, 2), min_df=5, sublinear_tf=True)
X_tr = vec2.fit_transform(tr["tokens"])
X_va = vec2.transform(va["tokens"])
print("vocab size:", len(vec2.vocabulary_))

t0 = time.time()
clf2 = LogisticRegression(class_weight="balanced", max_iter=1000)
clf2.fit(X_tr, tr["label"])
print(f"trained in {time.time() - t0:.0f}s")

pred2 = clf2.predict(X_va)
labels = ["neg", "neu", "pos"]
print("\n", classification_report(va["label"], pred2, labels=labels, digits=3))
print(pd.DataFrame(confusion_matrix(va["label"], pred2, labels=labels), index=labels, columns=labels))

names = vec2.get_feature_names_out()
for i, c in enumerate(clf2.classes_):
    top = clf2.coef_[i].argsort()[-15:][::-1]
    print(f"\ntop pieces for {c}:", " | ".join(names[top]))

vocab size: 255730
trained in 43s

               precision    recall  f1-score   support

         neg      0.564     0.653     0.605      4720
         neu      0.402     0.571     0.471      7281
         pos      0.923     0.797     0.855     28023

    accuracy                          0.739     40024
   macro avg      0.629     0.674     0.644     40024
weighted avg      0.786     0.739     0.756     40024

      neg   neu    pos
neg  3081  1347    292
neu  1543  4156   1582
pos   838  4845  22340

top pieces for neg: . ▁كل | ▁سيئة | ▁. ▁كل | ▁سيء | ▁غير | ▁سيئه | ▁سئ | ▁لا | ▁سيئ | سيئ | ▁عدم | ▁لايستحق | سيء | ▁خايس | ▁زباله

top pieces for neu: . ▁كل | ▁ما ”. | شكرا ”. | مقبول | ▁القيمة ”. | ▁فقط ”. | ▁عدم ▁سرعة | ▁جيد | فندق ▁جيد | ▁عادي | فندق ▁عادي | ▁والنظافة . | متوسط | ▁جميع ▁الفنادق | ▁متوسطه

top pieces for pos: ▁رائع | ▁ممتاز | ▁ممتازة | . ▁لا | ▁ممتازه | ▁راقي | ▁رائعة | ممتاز | ▁كل | رائع | ▁وسرعة | ▁جميل | ▁والهدوء | ▁رائعه | ▁كل ▁شي


In [ ]:
pred_tr = clf2.predict(X_tr)
rep_tr = classification_report(tr["label"], pred_tr, labels=labels, output_dict=True)
rep_va = classification_report(va["label"], pred2, labels=labels, output_dict=True)

rows = labels + ["macro avg"]
print(pd.DataFrame({
    "train f1": [rep_tr[r]["f1-score"] for r in rows],
    "val f1":   [rep_va[r]["f1-score"] for r in rows],
}, index=rows).round(3))

           train f1  val f1
neg           0.818   0.605
neu           0.687   0.471
pos           0.889   0.855
macro avg     0.798   0.644


In [ ]:
results = []
for C in [1, 0.3, 0.1, 0.03]:
    t0 = time.time()
    m = LogisticRegression(C=C, class_weight="balanced", max_iter=1000)
    m.fit(X_tr, tr["label"])
    f_tr = f1_score(tr["label"], m.predict(X_tr), labels=labels, average=None)
    f_va = f1_score(va["label"], m.predict(X_va), labels=labels, average=None)
    results.append({"C": C,
                    "train macro": f_tr.mean(), "val macro": f_va.mean(),
                    "gap": f_tr.mean() - f_va.mean(),
                    "train neg": f_tr[0], "val neg": f_va[0],
                    "sec": time.time() - t0})

print(pd.DataFrame(results).round(3).to_string(index=False))

   C  train macro  val macro   gap  train neg  val neg    sec
1.00        0.798      0.644 0.154      0.818    0.605 41.643
0.30        0.735      0.650 0.086      0.746    0.623 44.265
0.10        0.683      0.641 0.041      0.680    0.619 22.884
0.03        0.645      0.626 0.019      0.631    0.600 12.202


In [ ]:
lens = data.loc[data["split"] == "train", "tokens"].str.split().str.len()
print(lens.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))
for L in [128, 256, 512]:
    print(L, round((lens > L).mean(), 4))

count    327264.0
mean         27.0
std          28.0
min           1.0
50%          19.0
90%          56.0
95%          76.0
99%         134.0
max         839.0
Name: tokens, dtype: float64
128 0.0114
256 0.0012
512 0.0001


In [ ]:
importlib.reload(rp)
pad_id = sp2.get_piece_size()
seqs = rp.encode_ids(data[data["split"] == "train"].head(3), sp2)
ids, mask = rp.make_batch(seqs, pad_id)
print(pad_id)
print([len(s) for s in seqs])
print(ids.shape)
print(ids)
print(mask.int())

16000
[118, 7, 7]
torch.Size([3, 118])
tensor([[   24,    30,   651,  3436,  2454,  3153,   648,    19,    48,  1471,
           135,   750,   278,   523,    40, 12473,    88,  1155,   123,   907,
          4653,   688,   964,    20,   124,   332, 13662,  1099,  1296,   482,
         15876,  5589,  1348,  3354,  2441,    83,  5350,   629,   175,    92,
           141,    54,   691, 15868,   880,   674,    40,   651,   785,  1585,
          1967,  1254, 12745,  5492,    88,    23,  2984,    65, 14576,  1122,
          7394,   616, 15854,   201,   238,   437,  2000,  1471,    20,  3534,
            48,   135,    40,  2050,   368,    40,   908,  8463,  1649,  3831,
            40,   405,    20,   220, 15865,  1301,  1471,    29,  5286,  1735,
           989,    44,  6483,   129, 15436,  1642,   423, 15869,  1352, 15900,
          1014,    40,   437,  2439,  2921,  9062,  5807,   233,   195,   238,
          7125,   943,   195,  4879,    58,   495,   332,    80],
        [   24,   418,    

In [ ]:
importlib.reload(rp)
tr_ids = rp.encode_ids(data[data["split"] == "train"], sp2)
lens = np.array([len(s) for s in tr_ids])

def pad_share(batches):
    real = sum(lens[b].sum() for b in batches)
    total = sum(len(b) * lens[b].max() for b in batches)
    return 1 - real / total

rng = np.random.default_rng(0)
perm = rng.permutation(len(lens))
random_batches = [perm[i:i + 256] for i in range(0, len(perm), 256)]
bucket_batches = rp.length_batches(lens, 256, seed=0)
print(len(random_batches), len(bucket_batches))
print(round(pad_share(random_batches), 3), round(pad_share(bucket_batches), 3))

1279 1279
0.867 0.051


In [ ]:
import torch
importlib.reload(rp)
torch.manual_seed(0)
model = rp.MeanPoolClassifier(vocab_size=pad_id + 1, pad_id=pad_id)
b = bucket_batches[0]
ids, mask = rp.make_batch([tr_ids[i] for i in b], pad_id)
logits = model(ids, mask)
print(ids.shape, mask.shape)
print(logits.shape)
print(sum(p.numel() for p in model.parameters()))

torch.Size([256, 12]) torch.Size([256, 12])
torch.Size([256, 3])
1600403


In [ ]:
importlib.reload(rp)
tr = data[data["split"] == "train"]
va = data[data["split"] == "val"]
tr_y = tr["label"].map(rp.LABEL_IDS).to_numpy()
va_y = va["label"].map(rp.LABEL_IDS).to_numpy()
va_ids = rp.encode_ids(va, sp2)
print(rp.class_weights(tr_y))

torch.manual_seed(0)
model = rp.MeanPoolClassifier(vocab_size=pad_id + 1, pad_id=pad_id)
t0 = time.time()
hist = rp.train_model(model, tr_ids, tr_y, va_ids, va_y, pad_id)
print(round(time.time() - t0), "sec")
torch.save(model.state_dict(), MODELS / "meanpool_seed0.pt")

tensor([2.5931, 1.6973, 0.4938])
{'epoch': 1, 'train loss': 0.9095370441258472, 'val macro': 0.5812844199880877}
{'epoch': 2, 'train loss': 0.7917153741638018, 'val macro': 0.5975913487417657}
{'epoch': 3, 'train loss': 0.7620460043860819, 'val macro': 0.6074064812458159}
{'epoch': 4, 'train loss': 0.7453641253120565, 'val macro': 0.6071180743916097}
{'epoch': 5, 'train loss': 0.7335976086323056, 'val macro': 0.6067468899199944}
70 sec


In [ ]:
vec_uni = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                          ngram_range=(1, 1), min_df=5, sublinear_tf=True)
X_tr_uni = vec_uni.fit_transform(tr["tokens"])
X_va_uni = vec_uni.transform(va["tokens"])
m_uni = LogisticRegression(C=0.3, class_weight="balanced", max_iter=1000)
m_uni.fit(X_tr_uni, tr["label"])
f = f1_score(va["label"], m_uni.predict(X_va_uni), labels=labels, average=None)
print(X_tr_uni.shape)
print(round(f.mean(), 3), f.round(3))

(327264, 16081)
0.619 [0.582 0.449 0.827]


In [ ]:
runs = [{"seed": 0, "val macro": hist["val macro"].max()}]
for seed in [1, 2]:
    torch.manual_seed(seed)
    m = rp.MeanPoolClassifier(vocab_size=pad_id + 1, pad_id=pad_id)
    h = rp.train_model(m, tr_ids, tr_y, va_ids, va_y, pad_id, seed=seed)
    runs.append({"seed": seed, "val macro": h["val macro"].max()})
    torch.save(m.state_dict(), MODELS / f"meanpool_seed{seed}.pt")
runs = pd.DataFrame(runs)
print(runs.round(3).to_string(index=False))
print(round(runs["val macro"].mean(), 3), round(runs["val macro"].min(), 3), round(runs["val macro"].max(), 3))

{'epoch': 1, 'train loss': 0.9090413827635949, 'val macro': 0.5798465425412379}
{'epoch': 2, 'train loss': 0.7924274976062112, 'val macro': 0.6049998082343319}
{'epoch': 3, 'train loss': 0.7626548454047438, 'val macro': 0.6113636448904652}
{'epoch': 4, 'train loss': 0.7455098100324944, 'val macro': 0.6079991093336757}
{'epoch': 5, 'train loss': 0.7341718983123341, 'val macro': 0.6022323809990507}
{'epoch': 1, 'train loss': 0.9105520457877341, 'val macro': 0.586060291502695}
{'epoch': 2, 'train loss': 0.7929141483570784, 'val macro': 0.6069204251223099}
{'epoch': 3, 'train loss': 0.7625429278258368, 'val macro': 0.6083212037234363}
{'epoch': 4, 'train loss': 0.7456968359622226, 'val macro': 0.6025736404140228}
{'epoch': 5, 'train loss': 0.7339804509850066, 'val macro': 0.6024962526571459}
 seed  val macro
    0      0.607
    1      0.611
    2      0.608
0.609 0.607 0.611


In [ ]:
importlib.reload(rp)
att_runs = []
for seed in [0, 1, 2]:
    torch.manual_seed(seed)
    m = rp.AttentionClassifier(vocab_size=pad_id + 1, pad_id=pad_id)
    h = rp.train_model(m, tr_ids, tr_y, va_ids, va_y, pad_id, seed=seed)
    att_runs.append({"seed": seed, "val macro": h["val macro"].max()})
    torch.save(m.state_dict(), MODELS / f"attention_seed{seed}.pt")
att_runs = pd.DataFrame(att_runs)
print(sum(p.numel() for p in m.parameters()))
print(att_runs.round(3).to_string(index=False))
print(round(att_runs["val macro"].mean(), 3), round(att_runs["val macro"].min(), 3), round(att_runs["val macro"].max(), 3))

{'epoch': 1, 'train loss': 0.8923347890930721, 'val macro': 0.5777673084910558}
{'epoch': 2, 'train loss': 0.7784804044679496, 'val macro': 0.5983489390162626}
{'epoch': 3, 'train loss': 0.7463395988485431, 'val macro': 0.6109845687626536}
{'epoch': 4, 'train loss': 0.7281476087766949, 'val macro': 0.6179303212512056}
{'epoch': 5, 'train loss': 0.7147457870474726, 'val macro': 0.6150947028642596}
{'epoch': 6, 'train loss': 0.7052798794023893, 'val macro': 0.6164388952570673}
{'epoch': 1, 'train loss': 0.897386524740687, 'val macro': 0.5729483660795943}
{'epoch': 2, 'train loss': 0.7832128498413654, 'val macro': 0.5985382702694891}
{'epoch': 3, 'train loss': 0.7493884949445422, 'val macro': 0.617787057694847}
{'epoch': 4, 'train loss': 0.7296038076035307, 'val macro': 0.6172247476926706}
{'epoch': 5, 'train loss': 0.7165890853402901, 'val macro': 0.6196207348015971}
{'epoch': 6, 'train loss': 0.706580880900055, 'val macro': 0.6010090310989988}
{'epoch': 7, 'train loss': 0.69886906339682

In [ ]:
importlib.reload(rp)
conv_runs = []
for seed in [0, 1, 2]:
    torch.manual_seed(seed)
    m = rp.ConvAttentionClassifier(vocab_size=pad_id + 1, pad_id=pad_id)
    h = rp.train_model(m, tr_ids, tr_y, va_ids, va_y, pad_id, seed=seed)
    conv_runs.append({"seed": seed, "val macro": h["val macro"].max()})
    torch.save(m.state_dict(), MODELS / f"conv_attention_seed{seed}.pt")
conv_runs = pd.DataFrame(conv_runs)
print(sum(p.numel() for p in m.parameters()))
print(conv_runs.round(3).to_string(index=False))
print(round(conv_runs["val macro"].mean(), 3), round(conv_runs["val macro"].min(), 3), round(conv_runs["val macro"].max(), 3))

{'epoch': 1, 'train loss': 0.8202708340511767, 'val macro': 0.6245083474457669}
{'epoch': 2, 'train loss': 0.7169896410371104, 'val macro': 0.6364563016270025}
{'epoch': 3, 'train loss': 0.678512751461649, 'val macro': 0.6444687090768998}
{'epoch': 4, 'train loss': 0.6500202129170891, 'val macro': 0.6502412891445398}
{'epoch': 5, 'train loss': 0.6221726593633685, 'val macro': 0.6443103149850616}
{'epoch': 6, 'train loss': 0.5934625834247007, 'val macro': 0.6379000694080443}
{'epoch': 1, 'train loss': 0.8171336769742957, 'val macro': 0.6145246399937837}
{'epoch': 2, 'train loss': 0.7193653358617719, 'val macro': 0.6402736671828831}
{'epoch': 3, 'train loss': 0.6818347553872545, 'val macro': 0.6493490277466983}
{'epoch': 4, 'train loss': 0.6520554245223488, 'val macro': 0.6465685380833109}
{'epoch': 5, 'train loss': 0.6252306932005188, 'val macro': 0.6470141731863069}
{'epoch': 1, 'train loss': 0.8181434670792678, 'val macro': 0.6123083263207486}
{'epoch': 2, 'train loss': 0.719466432583

In [ ]:
importlib.reload(rp)
drop_runs = []
for seed in [0, 1, 2]:
    torch.manual_seed(seed)
    m = rp.ConvAttentionClassifier(vocab_size=pad_id + 1, pad_id=pad_id, dropout=0.3)
    h = rp.train_model(m, tr_ids, tr_y, va_ids, va_y, pad_id, seed=seed)
    drop_runs.append({"seed": seed, "val macro": h["val macro"].max(),
                      "best epoch": int(h["val macro"].idxmax()) + 1})
    torch.save(m.state_dict(), MODELS / f"conv_attention_drop03_seed{seed}.pt")
drop_runs = pd.DataFrame(drop_runs)
print(sum(p.numel() for p in m.parameters()))
print(drop_runs.round(3).to_string(index=False))
print(round(drop_runs["val macro"].mean(), 3), round(drop_runs["val macro"].min(), 3), round(drop_runs["val macro"].max(), 3))

{'epoch': 1, 'train loss': 0.8364884063355253, 'val macro': 0.6228016826594261}
{'epoch': 2, 'train loss': 0.7298625785257502, 'val macro': 0.6370860176990067}
{'epoch': 3, 'train loss': 0.6949228078106509, 'val macro': 0.6492903697622677}
{'epoch': 4, 'train loss': 0.6725085095609984, 'val macro': 0.654370552315817}
{'epoch': 5, 'train loss': 0.653234401289594, 'val macro': 0.6518334200418179}
{'epoch': 6, 'train loss': 0.6360546759616318, 'val macro': 0.6548344669090839}
{'epoch': 7, 'train loss': 0.6206416569229817, 'val macro': 0.6346764800012511}
{'epoch': 8, 'train loss': 0.6056495111884056, 'val macro': 0.6449970966361602}
{'epoch': 1, 'train loss': 0.8341112463101504, 'val macro': 0.6178746412389675}
{'epoch': 2, 'train loss': 0.7327736653959436, 'val macro': 0.6429234000285532}
{'epoch': 3, 'train loss': 0.697633812891909, 'val macro': 0.6562050214433036}
{'epoch': 4, 'train loss': 0.6732323554775123, 'val macro': 0.6489621160681266}
{'epoch': 5, 'train loss': 0.65438084664950

In [ ]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

2.14.0+cu126
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [ ]:
importlib.reload(rp)
device = torch.device("cuda")
torch.manual_seed(0)
m = rp.ConvAttentionClassifier(vocab_size=pad_id + 1, pad_id=pad_id, dropout=0.3).to(device)
t0 = time.time()
h = rp.train_model(m, tr_ids, tr_y, va_ids, va_y, pad_id, seed=0)
print(round(time.time() - t0), "sec")
print(round(h["val macro"].max(), 3))

{'epoch': 1, 'train loss': 0.8363076138615293, 'val macro': 0.6217667490855873}
{'epoch': 2, 'train loss': 0.7298725055636485, 'val macro': 0.6379081463597368}
{'epoch': 3, 'train loss': 0.6947674525498622, 'val macro': 0.6503840959310919}
{'epoch': 4, 'train loss': 0.6725615421686166, 'val macro': 0.6564920843866477}
{'epoch': 5, 'train loss': 0.6539696598666102, 'val macro': 0.6517418730328531}
{'epoch': 6, 'train loss': 0.6359957432285297, 'val macro': 0.6521087867491636}
111 sec
0.656


In [ ]:
importlib.reload(rp)
args = dict(tr_seqs=tr_ids, tr_y=tr_y, va_seqs=va_ids, va_y=va_y, pad_id=pad_id,
            models_dir=MODELS, device=device)
emb_only = rp.run_seeds(lambda: rp.ConvAttentionClassifier(pad_id + 1, pad_id, emb_dropout=0.3),
                        "conv_att_embdrop03", **args)
both = rp.run_seeds(lambda: rp.ConvAttentionClassifier(pad_id + 1, pad_id, dropout=0.3, emb_dropout=0.3),
                    "conv_att_bothdrop03", **args)
res = pd.concat([emb_only, both])
print(res.round(3).to_string(index=False))
print(res.groupby("model")["val macro"].agg(["mean", "min", "max"]).round(3))

{'epoch': 1, 'train loss': 0.8654338184474931, 'val macro': 0.6195221491540619}
{'epoch': 2, 'train loss': 0.7621206200762766, 'val macro': 0.6344718328546801}
{'epoch': 3, 'train loss': 0.7289595004399524, 'val macro': 0.6470639088213948}
{'epoch': 4, 'train loss': 0.708949937957513, 'val macro': 0.6541072913165656}
{'epoch': 5, 'train loss': 0.693577209257527, 'val macro': 0.6538867522569631}
{'epoch': 6, 'train loss': 0.6814393242678125, 'val macro': 0.6541892054704318}
{'epoch': 7, 'train loss': 0.6696096986249546, 'val macro': 0.6457714912554753}
{'epoch': 8, 'train loss': 0.6593988602772433, 'val macro': 0.6546453651494869}
{'epoch': 9, 'train loss': 0.6524718591308295, 'val macro': 0.6576888875460476}
{'epoch': 10, 'train loss': 0.6445042984604754, 'val macro': 0.6549846706858027}
{'epoch': 1, 'train loss': 0.8593435516930447, 'val macro': 0.6161791531664156}
{'epoch': 2, 'train loss': 0.762730466587471, 'val macro': 0.6397645892258619}
{'epoch': 3, 'train loss': 0.7307444200483

In [ ]:
importlib.reload(rp)
gru = rp.run_seeds(lambda: rp.GRUAttentionClassifier(pad_id + 1, pad_id, emb_dropout=0.3),
                   "gru_att_embdrop03", **args)
print(sum(p.numel() for p in rp.GRUAttentionClassifier(pad_id + 1, pad_id).parameters()))
print(gru.round(3).to_string(index=False))
print(gru["val macro"].agg(["mean", "min", "max"]).round(3))

{'epoch': 1, 'train loss': 0.80963066477597, 'val macro': 0.636194512720477}
{'epoch': 2, 'train loss': 0.7202240425703217, 'val macro': 0.6470691536671761}
{'epoch': 3, 'train loss': 0.694212045314737, 'val macro': 0.663228836330401}
{'epoch': 4, 'train loss': 0.6769175863608687, 'val macro': 0.6700297694350849}
{'epoch': 5, 'train loss': 0.6633524179715058, 'val macro': 0.6659478728377747}
{'epoch': 6, 'train loss': 0.651937395943287, 'val macro': 0.6679138730098552}
{'epoch': 1, 'train loss': 0.8085541517034914, 'val macro': 0.6257461613868646}
{'epoch': 2, 'train loss': 0.7195027675157281, 'val macro': 0.6550597532904393}
{'epoch': 3, 'train loss': 0.6944663589203756, 'val macro': 0.6676550002443626}
{'epoch': 4, 'train loss': 0.6766850119393814, 'val macro': 0.6673646444874938}
{'epoch': 5, 'train loss': 0.6640602413703703, 'val macro': 0.6680379209502069}
{'epoch': 6, 'train loss': 0.6516992097564418, 'val macro': 0.6657085442459123}
{'epoch': 7, 'train loss': 0.6416232110270427,

In [ ]:
importlib.reload(rp)
sa = rp.run_seeds(lambda: rp.ConvSelfAttentionClassifier(pad_id + 1, pad_id, emb_dropout=0.3),
                  "conv_selfatt_embdrop03", **args)
print(sum(p.numel() for p in rp.ConvSelfAttentionClassifier(pad_id + 1, pad_id).parameters()))
print(sa.round(3).to_string(index=False))
print(sa["val macro"].agg(["mean", "min", "max"]).round(3))

{'epoch': 1, 'train loss': 0.8496466134957604, 'val macro': 0.6186187731136656}
{'epoch': 2, 'train loss': 0.7561589749698429, 'val macro': 0.6212749757486395}
{'epoch': 3, 'train loss': 0.7252256022912847, 'val macro': 0.6509959636248894}
{'epoch': 4, 'train loss': 0.7054892547601267, 'val macro': 0.6560568390087592}
{'epoch': 5, 'train loss': 0.6899399406704558, 'val macro': 0.6582171618410204}
{'epoch': 6, 'train loss': 0.6789489340325238, 'val macro': 0.6496189621918864}
{'epoch': 7, 'train loss': 0.6671968904119253, 'val macro': 0.6603642923381334}
{'epoch': 8, 'train loss': 0.6574434964633751, 'val macro': 0.6620026332605566}
{'epoch': 9, 'train loss': 0.6504969760226261, 'val macro': 0.657534418927538}
{'epoch': 10, 'train loss': 0.6431149287436894, 'val macro': 0.6579839234215684}
{'epoch': 1, 'train loss': 0.8485177349715542, 'val macro': 0.5920166311874694}
{'epoch': 2, 'train loss': 0.7594092672225771, 'val macro': 0.6455220822884234}
{'epoch': 3, 'train loss': 0.72899604248

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
names = ["neg", "neu", "pos"]

lr = LogisticRegression(C=0.3, class_weight="balanced", max_iter=1000).fit(X_tr, tr["label"])
base_pred = pd.Series(lr.predict(X_va)).map(rp.LABEL_IDS).to_numpy()

gru_preds = {}
for seed in [0, 1, 2]:
    g = rp.GRUAttentionClassifier(pad_id + 1, pad_id, emb_dropout=0.3).to(device)
    g.load_state_dict(torch.load(MODELS / f"gru_att_embdrop03_seed{seed}.pt"))
    gru_preds[seed] = rp.predict(g, va_ids, pad_id)

print("BASELINE")
print(classification_report(va_y, base_pred, target_names=names, digits=3))
print(confusion_matrix(va_y, base_pred))
print("GRU seed 0")
print(classification_report(va_y, gru_preds[0], target_names=names, digits=3))
print(confusion_matrix(va_y, gru_preds[0]))
print("GRU per-class F1, all seeds")
print(pd.DataFrame({s: f1_score(va_y, p, average=None) for s, p in gru_preds.items()},
                   index=names).round(3))

BASELINE
              precision    recall  f1-score   support

         neg      0.578     0.676     0.623      4720
         neu      0.398     0.599     0.478      7281
         pos      0.930     0.780     0.848     28023

    accuracy                          0.735     40024
   macro avg      0.635     0.685     0.650     40024
weighted avg      0.791     0.735     0.754     40024

[[ 3193  1258   269]
 [ 1532  4364  1385]
 [  803  5355 21865]]
GRU seed 0
              precision    recall  f1-score   support

         neg      0.626     0.675     0.650      4720
         neu      0.424     0.583     0.491      7281
         pos      0.923     0.821     0.869     28023

    accuracy                          0.761     40024
   macro avg      0.658     0.693     0.670     40024
weighted avg      0.797     0.761     0.774     40024

[[ 3188  1261   271]
 [ 1383  4243  1655]
 [  519  4493 23011]]
GRU per-class F1, all seeds
         0      1      2
neg  0.650  0.658  0.651
neu  0.491  

Session start

In [1]:
import importlib, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import reviews_pipeline as rp
importlib.reload(rp)
DATA = Path("data/hard")
MODELS = Path("models")
data = rp.load_or_build(DATA / "unbalanced-reviews.txt", DATA / "reviews_clean.parquet",
                        DATA / "site_labels.json")
print(data.shape, data["split"].value_counts().to_dict())

c:\Users\Asus\Desktop\eng\ai_control_projects\.venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(409312, 11) {'train': 327264, 'test': 42024, 'val': 40024}


In [2]:
from transformers import AutoTokenizer
bert_name = "aubmindlab/bert-base-arabertv02"
btok = AutoTokenizer.from_pretrained(bert_name)
tr_texts = data.loc[data["split"] == "train", "review_clean"].tolist()
blens = pd.Series([len(x) for x in btok(tr_texts, add_special_tokens=True)["input_ids"]])
print(blens.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))
for L in [64, 128, 256]:
    print(L, round((blens > L).mean(), 4))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (622 > 512). Running this sequence through the model will result in indexing errors


count    327264.0
mean         32.4
std          30.3
min           4.0
50%          24.0
90%          64.0
95%          85.0
99%         147.4
max         835.0
dtype: float64
64 0.0998
128 0.0153
256 0.0015


In [3]:
sp2 = rp.train_tokenizer(data.loc[data["split"] == "train", "review_clean"], MODELS / "spm_reviews_16k_v2")
sample = tr_texts[:20000]
chars = sum(len(t.replace(" ", "")) for t in sample)
ours = sum(len(sp2.encode(t)) for t in sample)
bert = sum(len(x) for x in btok(sample, add_special_tokens=False)["input_ids"])
print(round(chars / ours, 2), round(chars / bert, 2))

3.76 3.36


In [4]:
from transformers import AutoModelForSequenceClassification
device = torch.device("cuda")
bert = AutoModelForSequenceClassification.from_pretrained(bert_name, num_labels=3).to(device)
opt = torch.optim.AdamW(bert.parameters(), lr=2e-5)
long_idx = np.argsort(blens.to_numpy())[-32:]
batch = btok([tr_texts[i] for i in long_idx], truncation=True, max_length=256,
             padding=True, return_tensors="pt").to(device)
y = torch.zeros(32, dtype=torch.long, device=device)
torch.cuda.reset_peak_memory_stats()
with torch.autocast("cuda", dtype=torch.float16):
    out = bert(**batch, labels=y)
out.loss.backward()
opt.step()
print(batch["input_ids"].shape)
print(round(torch.cuda.max_memory_allocated() / 1e9, 2), "GB")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 740.48it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

torch.Size([32, 256])
3.91 GB


In [5]:
from transformers import AutoTokenizer
importlib.reload(rp)
device = torch.device("cuda")

bert_name = "aubmindlab/bert-base-arabertv02"
btok = AutoTokenizer.from_pretrained(bert_name)

tr_texts = data.loc[data["split"] == "train", "review_clean"].tolist()
tr_y = data.loc[data["split"] == "train", "label"].map(rp.LABEL_IDS).to_numpy()
tr_lens = np.array([len(x) for x in btok(tr_texts, add_special_tokens=True)["input_ids"]])
va = data[data["split"] == "val"]
va_texts = va["review_clean"].tolist()
va_y = va["label"].map(rp.LABEL_IDS).to_numpy()

for v in ["bert", "opt"]:
    globals().pop(v, None)
torch.cuda.empty_cache()

rows = []
for seed in [0, 1, 2]:
    path = MODELS / f"arabert_seed{seed}.pt"
    log = MODELS / f"arabert_seed{seed}_history.csv"
    if log.exists():
        hist = pd.read_csv(log)
    else:
        model, hist = rp.train_bert(bert_name, btok, tr_texts, tr_y, tr_lens,
                                    va_texts, va_y, device, seed=seed, max_len=128)
        torch.save(model.state_dict(), path)
        hist.to_csv(log, index=False)
        del model
        torch.cuda.empty_cache()
    rows.append({"seed": seed, "val macro": hist["val macro"].max(),
                 "best epoch": int(hist["val macro"].idxmax()) + 1})
    print(hist.round(3).to_string(index=False))

bert_runs = pd.DataFrame(rows)
print(bert_runs.round(3).to_string(index=False))
print(bert_runs["val macro"].agg(["mean", "min", "max"]).round(3))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (622 > 512). Running this sequence through the model will result in indexing errors
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1208.69it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISS

epoch 1 batch 2000/10227 4.7 min
epoch 1 batch 4000/10227 9.3 min
epoch 1 batch 6000/10227 13.4 min
epoch 1 batch 8000/10227 17.6 min
epoch 1 batch 10000/10227 21.8 min
{'epoch': 1, 'train loss': 0.7059471918271996, 'val macro': 0.6816790496415908, 'min': 22.663406948248547}
epoch 2 batch 2000/10227 26.9 min
epoch 2 batch 4000/10227 31.0 min
epoch 2 batch 6000/10227 35.4 min
epoch 2 batch 8000/10227 40.1 min
epoch 2 batch 10000/10227 44.7 min
{'epoch': 2, 'train loss': 0.6258802933637915, 'val macro': 0.701927714372475, 'min': 45.56575468381246}
epoch 3 batch 2000/10227 49.7 min
epoch 3 batch 4000/10227 53.9 min
epoch 3 batch 6000/10227 58.0 min
epoch 3 batch 8000/10227 62.4 min
epoch 3 batch 10000/10227 66.8 min
{'epoch': 3, 'train loss': 0.5728603094419893, 'val macro': 0.6952056725603083, 'min': 67.68597439130147}
 epoch  train loss  val macro    min
     1       0.706      0.682 22.663
     2       0.626      0.702 45.566
     3       0.573      0.695 67.686


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 612.42it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

epoch 1 batch 2000/10227 4.1 min
epoch 1 batch 4000/10227 8.2 min
epoch 1 batch 6000/10227 12.4 min
epoch 1 batch 8000/10227 16.7 min
epoch 1 batch 10000/10227 21.0 min
{'epoch': 1, 'train loss': 0.7059229542587327, 'val macro': 0.6910273005828792, 'min': 21.83197537660599}
epoch 2 batch 2000/10227 26.0 min
epoch 2 batch 4000/10227 30.3 min
epoch 2 batch 6000/10227 34.5 min
epoch 2 batch 8000/10227 38.8 min
epoch 2 batch 10000/10227 43.1 min
{'epoch': 2, 'train loss': 0.6244598934424171, 'val macro': 0.6913121824778701, 'min': 43.91837418476741}
epoch 3 batch 2000/10227 48.2 min
epoch 3 batch 4000/10227 52.4 min
epoch 3 batch 6000/10227 56.6 min
epoch 3 batch 8000/10227 60.8 min
epoch 3 batch 10000/10227 65.1 min
{'epoch': 3, 'train loss': 0.5711798456491025, 'val macro': 0.6956391699288487, 'min': 65.92358955144883}
 epoch  train loss  val macro    min
     1       0.706      0.691 21.832
     2       0.624      0.691 43.918
     3       0.571      0.696 65.924


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 801.67it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

epoch 1 batch 2000/10227 4.2 min
epoch 1 batch 4000/10227 8.3 min
epoch 1 batch 6000/10227 12.4 min
epoch 1 batch 8000/10227 16.5 min
epoch 1 batch 10000/10227 20.7 min
{'epoch': 1, 'train loss': 0.7066289428754485, 'val macro': 0.6926765699047844, 'min': 21.593904423713685}
epoch 2 batch 2000/10227 25.7 min
epoch 2 batch 4000/10227 29.8 min
epoch 2 batch 6000/10227 33.9 min
epoch 2 batch 8000/10227 38.0 min
epoch 2 batch 10000/10227 42.1 min
{'epoch': 2, 'train loss': 0.6236629513665776, 'val macro': 0.6937823039800078, 'min': 42.97781042655309}
epoch 3 batch 2000/10227 47.1 min
epoch 3 batch 4000/10227 51.2 min
epoch 3 batch 6000/10227 55.4 min
epoch 3 batch 8000/10227 59.5 min
epoch 3 batch 10000/10227 63.6 min
{'epoch': 3, 'train loss': 0.5724518325679216, 'val macro': 0.6985565669988736, 'min': 64.47838807900747}
 epoch  train loss  val macro    min
     1       0.707      0.693 21.594
     2       0.624      0.694 42.978
     3       0.572      0.699 64.478
 seed  val macro  best

In [2]:
importlib.reload(rp)
device = torch.device("cuda")
tr_y = data.loc[data["split"] == "train", "label"].map(rp.LABEL_IDS).to_numpy()
va = data[data["split"] == "val"]
va_y = va["label"].map(rp.LABEL_IDS).to_numpy()

sp2 = rp.train_tokenizer(data.loc[data["split"] == "train", "review_clean"], MODELS / "spm_reviews_16k_v2")
pad_id = sp2.get_piece_size()
tr_ids128 = rp.encode_ids(data[data["split"] == "train"], sp2, max_len=128)
va_ids128 = rp.encode_ids(va, sp2, max_len=128)

args128 = dict(tr_seqs=tr_ids128, tr_y=tr_y, va_seqs=va_ids128, va_y=va_y, pad_id=pad_id,
               models_dir=MODELS, device=device)
gru128 = rp.run_seeds(lambda: rp.GRUAttentionClassifier(pad_id + 1, pad_id, emb_dropout=0.3),
                      "gru_att_embdrop03_len128", **args128)
print(gru128.round(3).to_string(index=False))
print(gru128["val macro"].agg(["mean", "min", "max"]).round(3))

{'epoch': 1, 'train loss': 0.8099914911066762, 'val macro': 0.6347301205844039}
{'epoch': 2, 'train loss': 0.7211845618918271, 'val macro': 0.648458657023653}
{'epoch': 3, 'train loss': 0.6946767784288648, 'val macro': 0.6652037745841474}
{'epoch': 4, 'train loss': 0.6766857525592933, 'val macro': 0.6702011068621258}
{'epoch': 5, 'train loss': 0.6633900565358333, 'val macro': 0.6695692243952807}
{'epoch': 6, 'train loss': 0.6512412893360657, 'val macro': 0.6682519518578124}
{'epoch': 1, 'train loss': 0.8080687721045798, 'val macro': 0.6272695969303943}
{'epoch': 2, 'train loss': 0.7198232706893722, 'val macro': 0.6559481113787848}
{'epoch': 3, 'train loss': 0.6948502243316989, 'val macro': 0.6675675379898918}
{'epoch': 4, 'train loss': 0.6775458419242539, 'val macro': 0.6677241286836622}
{'epoch': 5, 'train loss': 0.6639150001499862, 'val macro': 0.6692954307590352}
{'epoch': 6, 'train loss': 0.6522035141885019, 'val macro': 0.6620548310703666}
{'epoch': 7, 'train loss': 0.642710270298

In [3]:
from transformers import AutoTokenizer
btok = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02")
lens = pd.DataFrame({
    "split": data["split"].to_numpy(),
    "label": data["label"].to_numpy(),
    "bert_len": [len(x) for x in btok(data["review_clean"].tolist(), add_special_tokens=True)["input_ids"]],
})
long = lens["bert_len"] > 128
print(long.groupby(lens["split"]).sum())
print(pd.crosstab(lens["label"], long, normalize="columns").round(3))

ex = data[long.to_numpy() & (data["label"] == "neg").to_numpy()].sample(1, random_state=0)
ids = btok(ex["review_clean"].iloc[0], add_special_tokens=True)["input_ids"]
print(ex["rating"].iloc[0], len(ids))
print(btok.decode(ids[1:127]))
print("---- cut ----")
print(btok.decode(ids[127:-1]))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (622 > 512). Running this sequence through the model will result in indexing errors


split
test      644
train    5017
val       610
Name: bert_len, dtype: int64
bert_len  False  True 
label                 
neg       0.126  0.293
neu       0.195  0.240
pos       0.678  0.467
2 188
حسنه الفندق الوحيد هو وجود كارفور اسفل الفندق وصيدلية ومحلات كثيره [UNK] lt ; [UNK] lt ; بالاضافه الي سرعه الاستجابه عند الطلب الاستقبال جيد [UNK] lt ; [UNK] lt ;. موقعه جدا غير مناسب فهو يقع في قلب الشارقه عند البحيره [UNK] lt ; يعني اذا تبي تطلع لازم تنتظر اكثر من ربع ساعه في الدخول والخروج بسبب الزحمه الخانقه الممله جدا [UNK] lt ; بالاضافه عدم وجود مواقف [UNK] lt ; العامل يشيل السياره نقطه مهمه [UNK] lt ; اخذو مني مبلغ زيتده ٢٠٠ درهم ولما سالتهم قالوا بنرجعه
---- cut ----
لك من تعمل تشيك اوت [UNK] lt ; والي الان ما رجعوا لي المبلغ غرف الفندق بارده جدا برغم من اطفا التكييف [UNK] lt ; اتصلت ع الاستقبال قالوا عندهم عطل في بعض الغرف وتم العطل طوال فتره السكن يعني التكييف شغال ٢٤ ساعه [UNK] lt ; [UNK] lt ;


In [4]:
has = data["review_clean"].str.contains("&lt;", regex=False)
print(has.sum(), round(has.mean(), 4))
print(data.loc[has, "label"].value_counts(normalize=True).round(3))
ent = data["review_clean"].str.extractall(r"(&[a-z]+;)")[0].value_counts()
print(ent.head(10))

62 0.0002
label
pos    0.548
neu    0.323
neg    0.129
Name: proportion, dtype: float64
0
&amp;     473
&gt;      126
&lt;      116
&nbsp;     45
Name: count, dtype: int64


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix
importlib.reload(rp)
device = torch.device("cuda")
bert_name = "aubmindlab/bert-base-arabertv02"
btok = AutoTokenizer.from_pretrained(bert_name)
va = data[data["split"] == "val"].reset_index(drop=True)
va_texts = va["review_clean"].tolist()
va_y = va["label"].map(rp.LABEL_IDS).to_numpy()

model = AutoModelForSequenceClassification.from_pretrained(bert_name, num_labels=3)
model.load_state_dict(torch.load(MODELS / "arabert_seed0.pt"))
model.to(device)
va_pred = rp.predict_bert(model, btok, va_texts, max_len=128)
names = ["neg", "neu", "pos"]
print(confusion_matrix(va_y, va_pred))

va["pred"] = [names[i] for i in va_pred]
AUD = DATA / "audit"
to_label = AUD / "errors_to_label.csv"
if not to_label.exists():
    parts = []
    for t in names:
        for p in names:
            cell = va[(va["label"] == t) & (va["pred"] == p)]
            n = 10 if t == p else 15
            parts.append(cell.sample(min(n, len(cell)), random_state=0))
    audit = pd.concat(parts).sample(frac=1, random_state=0).reset_index(drop=True)
    audit["item"] = range(len(audit))
    audit[["item", "review_clean"]].assign(my_label="", note="").to_csv(
        to_label, index=False, encoding="utf-8-sig")
    audit[["item", "label", "pred", "rating"]].to_csv(AUD / "errors_key.csv", index=False)
    print(len(audit), "reviews saved")
else:
    print("errors_to_label.csv exists -> not overwritten")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 700.89it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

[[ 3393  1119   208]
 [ 1314  4660  1307]
 [  311  4537 23175]]
120 reviews saved


In [6]:
path = DATA / "audit" / "errors_to_label.csv"
lab = pd.read_csv(path, encoding="utf-8-sig", keep_default_na=False)
for i in lab.index[lab["my_label"] == ""]:
    print(f"\nleft: {(lab['my_label'] == '').sum()} / {len(lab)}  |  p / n / m (+ space + note)  |  q = stop")
    print(lab.at[i, "review_clean"])
    ans = input("> ").strip()
    if ans == "q":
        break
    if not ans or ans[0] not in "pnm":
        continue
    lab.at[i, "my_label"] = ans[0]
    lab.at[i, "note"] = ans[2:].strip()
    lab.to_csv(path, index=False, encoding="utf-8-sig")
print("done:", (lab["my_label"] != "").sum(), "/", len(lab))


left: 120 / 120  |  p / n / m (+ space + note)  |  q = stop
. السرير نوعا مريح لكن التوسعه تجعل الفندق بعيد عن الحرم والباصات ايضا تنزلك بعيد عن الحرم والخدمة لابد ان تطلبها يعني لا يقومون بالنظافة وخلافه الا لم تطلب منهم والفندق لا يرقي الي أربعة نجوم

left: 119 / 120  |  p / n / m (+ space + note)  |  q = stop
“عمرة بيت الله”. لا يوجد. لا توجد باصات نضيفة و جيدة

left: 118 / 120  |  p / n / m (+ space + note)  |  q = stop
“الرياض”. . عدم وجود عازل لمغطس الحمام مما سبب في تناثر الماء في أرجاء الحمام

left: 117 / 120  |  p / n / m (+ space + note)  |  q = stop
. ١/سحب مبلغ الاقامة قبل الوصول  مع ان الدفع في مكان الاقامة .٢/ أنكر موظف الاستقبال سحب المبلغ من البطاقة الإئتمانية واصر علي سحب المبلغ مرة أخرى وبرر أن السبب يرجع للبنك وعند مخاطبة البنك أفاد بسحب الفندق للمبلغ وأعطانى رقم العملية .قلت لهم كل الفنادق تخصم مبلغ الإقامة ولا يتم بمطالبة المبلغ مرة أخرى سواكم. التأخر في تسجيل الدخول لمدة ٣ ساعات . ٤/ تعبئة الثلاجة بالمحرمات الخبيثة  دون خجل نسأل الله العافية والسلامة٥/ وصلت قلة ا

In [7]:
lab = pd.read_csv(DATA / "audit" / "errors_to_label.csv", encoding="utf-8-sig", keep_default_na=False)
key = pd.read_csv(DATA / "audit" / "errors_key.csv")
m = lab.merge(key, on="item")
m["me"] = m["my_label"].map({"p": "pos", "n": "neg", "m": "neu"})
m["error"] = m["label"] != m["pred"]

def verdict(r):
    if r["me"] == r["label"]:
        return "model wrong"
    if r["me"] == r["pred"]:
        return "label wrong"
    return "neither"

m["verdict"] = m.apply(verdict, axis=1)
err = m[m["error"]]
print(err["verdict"].value_counts())
print(pd.crosstab([err["label"], err["pred"]], err["verdict"]))
ok = m[~m["error"]]
print("agree on correct:", round((ok["me"] == ok["label"]).mean(), 2), "of", len(ok))

verdict
label wrong    58
model wrong    21
neither        11
Name: count, dtype: int64
verdict     label wrong  model wrong  neither
label pred                                   
neg   neu             9            6        0
      pos            10            0        5
neu   neg            11            4        0
      pos             9            6        0
pos   neg             9            3        3
      neu            10            2        3
agree on correct: 0.7 of 30


In [9]:
importlib.reload(rp)
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_fscore_support
device = torch.device("cuda")
tr = data[data["split"] == "train"]
va = data[data["split"] == "val"].reset_index(drop=True)
tr_y = tr["label"].map(rp.LABEL_IDS).to_numpy()
va_y = va["label"].map(rp.LABEL_IDS).to_numpy()
sp2 = rp.train_tokenizer(tr["review_clean"], MODELS / "spm_reviews_16k_v2")
pad_id = sp2.get_piece_size()
tr_ids = rp.encode_ids(tr, sp2)
va_ids = rp.encode_ids(va, sp2)

POS = rp.LABEL_IDS["pos"]
tr_y2 = (tr_y != POS).astype(int)
va_y2 = (va_y != POS).astype(int)
unhappy_f1 = lambda y, p: f1_score(y, p, average="binary", pos_label=1)

def unhappy_report(name, n_classes, unhappy_ids):
    rows = []
    for seed in [0, 1, 2]:
        g = rp.GRUAttentionClassifier(pad_id + 1, pad_id, n_classes=n_classes, emb_dropout=0.3).to(device)
        g.load_state_dict(torch.load(MODELS / f"{name}_seed{seed}.pt"))
        p2 = np.isin(rp.predict(g, va_ids, pad_id), unhappy_ids).astype(int)
        pr, rc, f1, _ = precision_recall_fscore_support(va_y2, p2, average="binary")
        rows.append({"seed": seed, "precision": pr, "recall": rc, "unhappy f1": f1})
    out = pd.DataFrame(rows)
    print(name)
    print(out.round(3).to_string(index=False))
    print(out[["precision", "recall", "unhappy f1"]].mean().round(3).to_dict())

unhappy_report("gru_att_embdrop03", n_classes=3, unhappy_ids=[0, 1])

args2 = dict(tr_seqs=tr_ids, tr_y=tr_y2, va_seqs=va_ids, va_y=va_y2, pad_id=pad_id,
             models_dir=MODELS, device=device)
rp.run_seeds(lambda: rp.GRUAttentionClassifier(pad_id + 1, pad_id, n_classes=2, emb_dropout=0.3),
             "gru_binary_unhappy", n_classes=2, score_fn=unhappy_f1,
             score_name="val unhappy f1", **args2)

unhappy_report("gru_binary_unhappy", n_classes=2, unhappy_ids=[1])

gru_att_embdrop03
 seed  precision  recall  unhappy f1
    0      0.668   0.840       0.744
    1      0.671   0.841       0.746
    2      0.668   0.842       0.745
{'precision': 0.669, 'recall': 0.841, 'unhappy f1': 0.745}
{'epoch': 1, 'train loss': 0.4616077220590278, 'val unhappy f1': 0.7283305982395942}
{'epoch': 2, 'train loss': 0.40168973722800116, 'val unhappy f1': 0.7405132870844232}
{'epoch': 3, 'train loss': 0.38400405148227995, 'val unhappy f1': 0.7454580372575379}
{'epoch': 4, 'train loss': 0.37287506482707705, 'val unhappy f1': 0.7477691618283131}
{'epoch': 5, 'train loss': 0.36359226790483457, 'val unhappy f1': 0.7480747205876546}
{'epoch': 6, 'train loss': 0.35672659318887956, 'val unhappy f1': 0.7478600390449016}
{'epoch': 7, 'train loss': 0.349874379602336, 'val unhappy f1': 0.7502868068833652}
{'epoch': 8, 'train loss': 0.34440192044906115, 'val unhappy f1': 0.7508717551336691}
{'epoch': 9, 'train loss': 0.3392344003923363, 'val unhappy f1': 0.7488041162519596}
{'epo

In [10]:
from sklearn.metrics import confusion_matrix

def unhappy_matrix(name, n_classes, unhappy_ids, seed=0):
    g = rp.GRUAttentionClassifier(pad_id + 1, pad_id, n_classes=n_classes, emb_dropout=0.3).to(device)
    g.load_state_dict(torch.load(MODELS / f"{name}_seed{seed}.pt"))
    p2 = np.isin(rp.predict(g, va_ids, pad_id), unhappy_ids).astype(int)
    cm = pd.DataFrame(confusion_matrix(va_y2, p2, labels=[1, 0]),
                      index=["true unhappy", "true happy"],
                      columns=["pred unhappy", "pred happy"])
    print(name, "seed", seed)
    print(cm, "\n")

unhappy_matrix("gru_att_embdrop03", 3, [0, 1])
unhappy_matrix("gru_binary_unhappy", 2, [1])

gru_att_embdrop03 seed 0
              pred unhappy  pred happy
true unhappy         10075        1926
true happy            5012       23011 

gru_binary_unhappy seed 0
              pred unhappy  pred happy
true unhappy          9690        2311
true happy            4119       23904 



In [12]:
txt = data["review_clean"].str.strip().str.strip('"').str.strip()
n_parts = txt.str.split(". ", regex=False).str.len()

print(n_parts.value_counts().sort_index().head(8))

for t in data.sample(5, random_state=0)["review_clean"]:
    print(repr(t[:300]))
    print("-" * 40)

review_clean
1     27658
2    167742
3    187739
4     13769
5      5583
6      2865
7      1575
8       896
Name: count, dtype: int64
'“Very good”. كل شي. '
----------------------------------------
'بوفيه السحور كان ممتاز و رخيص ..قربه من الحرم وايضا يوجد باصات ..الموظفين في الاستقبال جدا متعاونين ... لايوجد'
----------------------------------------
'“معانات لم اتوقعها في فندق اربع نجومً”. لا. لا شي'
----------------------------------------
'“اقامتي الممتعه”. النظافه وطاقم العمل. '
----------------------------------------
'“يامسافر للخبر..”. النظافة.. الافطار يأتيك في الغرفة وهوقليل ولكن متنوع..راقية وعصرية.. قريبة من بوابة رقم 5 للراشد مول ..فعلا خدمات فندقية... صغيرة لمن لديه أطفال..ومن تجربة شخصية من يريد شقق نظيفة تحيط بها الخدمات فعليه بأجنحة القصر الفندقية رحبة وواسعة وتقع في طريق الامير فيصل بن فهد والذي يتواجد'
----------------------------------------


In [13]:
groups = n_parts.clip(upper=5)

for n in [1, 2, 3, 4, 5]:
    print(f"===== {n} parts" + (" or more" if n == 5 else ""))
    for t in txt[groups == n].sample(2, random_state=0):
        print(repr(t[:300]))
        print("-" * 40)

===== 1 parts
'ممتاز جدا  والغرفه جميله  جدا.'
----------------------------------------
'موقع الهوتيل ممتاز قريب من المطاعم والاسواق.'
----------------------------------------
===== 2 parts
'“جيد”. النظافه جيده.'
----------------------------------------
'طاقم العمل. الاثاث قديم ازعاج'
----------------------------------------
===== 3 parts
'“استمتعت ..”. المكان رائع للاطفال ونظيف. التكيف سيئ ولم نستطيع النوم 😔'
----------------------------------------
'“جيد جدا”. الموقع ومعاملة الستاف. الاثاث قدبم'
----------------------------------------
===== 4 parts
'“مقبول لكن..”. اعجبنى الهدوء ... 20 دقيقة مشى الى الحرم اى بعيد شىء ما ...التاكسي 10 ريال من الحرم الى الفندق ..عدم توافر اكثر من سيارة باص للتوصيل الى الحرم تكون خاصة بالفندق .. فقط سيارة ميكروباص واحدة ..'
----------------------------------------
'“سيئه  وما حبيت الرحلة في هذا الفندق”. الغرفة روعه  حلو المكان  الفندق جيد لانه في بداية الجي بي أر. اولا لم يعجبني التأمين  وثانيا فندق  لا يستحق الخمس نجوم  ثالثا لا يوجد وأي فأي مجانا من ق

In [14]:
title = txt.str.extract(r"^“([^”]*)”", expand=False).str.strip()
print("has title:", round(title.notna().mean(), 3))

top = title.value_counts().head(15).index
tab = pd.crosstab(title[title.isin(top)], data["rating"], normalize="index").round(2)
tab["count"] = title.value_counts()[tab.index]
print(tab.sort_values("count", ascending=False))

has title: 0.563
rating           1     2     3     4     5  count
review_clean                                     
ممتاز         0.01  0.00  0.02  0.24  0.74   6277
جيد           0.00  0.03  0.37  0.54  0.06   4756
رائع          0.00  0.00  0.01  0.23  0.75   3452
جيد جدا       0.00  0.01  0.09  0.67  0.23   2117
فندق رائع     0.00  0.00  0.01  0.25  0.73   1609
مقبول         0.01  0.21  0.64  0.13  0.01   1476
جميل          0.00  0.01  0.07  0.50  0.42   1422
فندق ممتاز    0.01  0.00  0.01  0.26  0.72   1077
فندق جميل     0.00  0.00  0.05  0.46  0.48   1016
فندق جيد      0.00  0.02  0.26  0.65  0.08    728
ممتاز جدا     0.00  0.00  0.00  0.10  0.89    613
جيده          0.00  0.03  0.42  0.49  0.05    572
جيدة          0.00  0.03  0.37  0.52  0.07    520
جيد جداً      0.00  0.01  0.11  0.67  0.21    508
جميل جدا      0.01  0.00  0.01  0.27  0.71    496


In [15]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import precision_recall_fscore_support

bert_name = "aubmindlab/bert-base-arabertv02"
btok = AutoTokenizer.from_pretrained(bert_name)
va = data[data["split"] == "val"].reset_index(drop=True)
va_texts = va["review_clean"].tolist()
va_y2 = (va["label"] != "pos").astype(int).to_numpy()

rows = []
for seed in [0, 1, 2]:
    model = AutoModelForSequenceClassification.from_pretrained(bert_name, num_labels=3)
    model.load_state_dict(torch.load(MODELS / f"arabert_seed{seed}.pt"))
    model.to(device)
    p2 = np.isin(rp.predict_bert(model, btok, va_texts, max_len=128), [0, 1]).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(va_y2, p2, average="binary")
    rows.append({"seed": seed, "precision": pr, "recall": rc, "unhappy f1": f1})
    del model
    torch.cuda.empty_cache()

out = pd.DataFrame(rows)
print(out.round(3).to_string(index=False))
print(out[["precision", "recall", "unhappy f1"]].mean().round(3).to_dict())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 633.68it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

 seed  precision  recall  unhappy f1
    0      0.684   0.874       0.767
    1      0.677   0.880       0.765
    2      0.683   0.876       0.768
{'precision': 0.682, 'recall': 0.877, 'unhappy f1': 0.767}


In [17]:
importlib.reload(rp)

SEED = 0
model = AutoModelForSequenceClassification.from_pretrained(bert_name, num_labels=3)
model.load_state_dict(torch.load(MODELS / f"arabert_seed{SEED}.pt"))
model.to(device)
va_p = rp.bert_unhappy_probs(model, btok, va_texts)

rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    pred = va_p >= t
    tp = int((pred & (va_y2 == 1)).sum())
    fn = int((~pred & (va_y2 == 1)).sum())
    fp = int((pred & (va_y2 == 0)).sum())
    rows.append({"threshold": t, "caught": tp, "missed": fn, "false alarms": fp,
                 "flagged": tp + fp, "recall": tp / (tp + fn), "precision": tp / (tp + fp)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 877.17it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from

 threshold  caught  missed  false alarms  flagged  recall  precision
       0.1   11776     225         13716    25492   0.981      0.462
       0.2   11518     483          9798    21316   0.960      0.540
       0.3   11219     782          7630    18849   0.935      0.595
       0.4   10911    1090          6208    17119   0.909      0.637
       0.5   10578    1423          5156    15734   0.881      0.672
       0.6   10168    1833          4163    14331   0.847      0.710


In [18]:
from sklearn.metrics import f1_score, confusion_matrix

THRESHOLD = 0.4
te = data[data["split"] == "test"].reset_index(drop=True)
te_texts = te["review_clean"].tolist()
te_y = te["label"].map(rp.LABEL_IDS).to_numpy()
te_y2 = (te_y != rp.LABEL_IDS["pos"]).astype(int)

te_p = rp.bert_unhappy_probs(model, btok, te_texts)
pred = te_p >= THRESHOLD
tp = int((pred & (te_y2 == 1)).sum())
fn = int((~pred & (te_y2 == 1)).sum())
fp = int((pred & (te_y2 == 0)).sum())
print({"reviews": len(te), "caught": tp, "missed": fn, "false alarms": fp,
       "flagged": tp + fp, "recall": round(tp / (tp + fn), 3),
       "precision": round(tp / (tp + fp), 3),
       "unhappy f1": round(2 * tp / (2 * tp + fn + fp), 3)})

te_pred3 = rp.predict_bert(model, btok, te_texts, max_len=128)
print("3-class macro:", round(f1_score(te_y, te_pred3, average="macro"), 3))
print(confusion_matrix(te_y, te_pred3))

{'reviews': 42024, 'caught': 13537, 'missed': 1221, 'false alarms': 6643, 'flagged': 20180, 'recall': 0.917, 'precision': 0.671, 'unhappy f1': 0.775}
3-class macro: 0.706
[[ 4354  1448   223]
 [ 1613  5613  1507]
 [  342  4803 22121]]
